[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/capstone/facility_decision.ipynb)

# Should this site build on-site generation?
## REE 4301 / IE 5300 - Energy Systems Modeling
### The outro. Do this last.

Module 0 was an introduction. This is the other end of it.

In **M0B** you learned that choosing a boundary is the first modelling decision, and you saw one site - the Metroplex Industrial Park - answered two different ways at two different sizes. Then Modules 1 to 4 each closed with a short *At the Facility Scale* segment showing what that module's tools look like once the boundary drops.

This notebook is all of it at once. **One facility. One question.**

> **Should the Metroplex Industrial Park build on-site generation?**

### The design constraint: nothing new

There is no new machinery in this notebook. Not one component, argument or formula that has not already appeared on a slide or in an earlier notebook. If you find yourself thinking *I have not seen this before*, you have - and the markdown will tell you where.

That is deliberate. The point of a capstone is not to teach you a sixth thing. It is to show you that the five things you already have are enough to answer a real question end to end - which is exactly what you will be asked to do in your first job, and exactly what you should be able to describe in an interview. **Part 9 makes you write that description.**


In [1]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/capstone"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = ["highspy>=1.11,<2", "pypsa>=1.3,<2"]

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---



working directory: capstone


We need three libraries and nothing else. `warnings` is silenced because PyPSA is noisy about pandas dtypes on import; nothing here depends on the warnings.


In [2]:
import numpy as np
import pandas as pd
import pypsa
import warnings
warnings.filterwarnings('ignore')

SOLVER = 'highs'      # open source, no licence, no size cap
pd.set_option('display.width', 160)

print('pypsa', pypsa.__version__, '· solver', SOLVER)


pypsa 1.3.0 · solver highs


---
# Part 0 - The boundary, stated before anything else

Every model in this course had a boundary. Most of them had one you did not choose. This one you choose, and you state it first, because every number after this inherits it.

**Inside the boundary:** the Metroplex Industrial Park. Roughly 50 MW of warehousing, light manufacturing and a cold-storage tenant, north of Dallas. Its roof, its yard, its meter, its equipment, its bill.

**Outside the boundary:** ERCOT. The generation fleet. West Texas wind. What gets built in 2035. None of it is modelled here.

**Crossing the boundary,** and this is the list from M0B Part F:

| channel | how it reaches this site |
|---|---|
| a price signal | a downloaded hourly series, plus the tariff wrapped around it |
| coincident-peak exposure | ERCOT 4CP - four intervals a year set the transmission charge |
| interconnection capacity | a 600 MW agreement the site has never come close to using |
| marginal emissions | nobody has promised 24/7 carbon-free energy here, so this channel is dormant |

This is the site **before** the investors show up. In M0B it becomes a 500 MW data centre; Part 7 re-asks today's question at that size, and gets a different answer for a reason worth understanding.


### The site, as data

Everything a facility analyst is handed on day one: a tariff sheet, a roof survey, an interconnection agreement, and a quote. No forecasts, no scenarios - four documents.


In [3]:
# --- the site
ROOF_SQFT = 1_200_000.0    # usable roof and canopy area, from the survey
W_PER_SQFT = 10.0          # modern modules with row spacing on a flat roof
ICA_MW = 600.0             # the interconnection agreement, from M0B

# --- the tariff, from the utility's large commercial schedule
ENERGY_ADDER = 12.0        # $/MWh   retail margin, ancillaries, losses
DELIVERY_VOL = 18.0        # $/MWh   volumetric delivery
TRANS_4CP = 70.0           # $/kW-yr transmission, set by the 4CP average
DIST_DEMAND = 4.50         # $/kW-month distribution, on the peak

# --- the quotes
SOLAR_CAPEX = 1_100_000.0  # $/MW-dc installed, commercial rooftop
SOLAR_FOM = 16_000.0       # $/MW-yr
BATT_CAPEX_KWH = 300.0     # $/kWh - the same figure as Module 4 slide 23
BATT_HOURS = 2.0
RTE = 0.86                 # round trip, Module 4 slide 23

ROOF_MW = ROOF_SQFT * W_PER_SQFT / 1e6
DEMAND_PER_MW_YR = (TRANS_4CP + DIST_DEMAND * 12) * 1000.0

print(f'usable roof          {ROOF_MW:8.1f} MW-dc')
print(f'interconnection      {ICA_MW:8.0f} MW')
print(f'volumetric adders    ${ENERGY_ADDER + DELIVERY_VOL:8.2f} /MWh on top of the wholesale price')
print(f'demand charges       ${DEMAND_PER_MW_YR:8,.0f} /MW-yr on the peak')


usable roof              12.0 MW-dc
interconnection           600 MW
volumetric adders    $   30.00 /MWh on top of the wholesale price
demand charges       $ 124,000 /MW-yr on the peak


> **Note what that last line is.** `TRANS_4CP + DIST_DEMAND * 12` is a charge on your **highest single interval of the year**, not on your energy. It will turn out to be the entire reason this project happens. Watch it.


---
# Part 1 - What does the site pay now?
### Module 1's tools, at the facility scale

Module 1 forecast demand for populations and regions. This site does not forecast its demand - it **measures** it. What follows is the interval meter, reduced to two **representative days** exactly the way Module 1 slides 13 to 16 described: a summer weekday and a winter weekday, weighted to add up to a year.

Two days is a screening model, not a study. A real analysis uses all 8,760 hours. **Name that as an assumption in anything you hand in.**


In [4]:
hours = np.arange(24)

# summer weekday: cold storage and HVAC ride the afternoon; a day shift
park_summer = (28.0
               + 22.0 * np.exp(-((hours - 16) ** 2) / 22.0)
               + 4.0 * np.exp(-((hours - 8) ** 2) / 6.0))

# winter weekday: flatter, and the peak moves to the morning
park_winter = (25.0
               + 9.0 * np.exp(-((hours - 10) ** 2) / 26.0)
               + 5.0 * np.exp(-((hours - 18) ** 2) / 12.0))

W_SUMMER, W_WINTER = 165, 200      # days per year each one stands for
assert W_SUMMER + W_WINTER == 365

print(pd.DataFrame({'summer MW': park_summer.round(1),
                    'winter MW': park_winter.round(1)}).T.to_string())


             0     1     2     3     4     5     6     7     8     9     10    11    12    13    14    15    16    17    18    19    20    21    22    23
summer MW  28.0  28.0  28.0  28.1  28.3  29.0  30.3  31.9  33.2  33.8  34.3  36.0  38.9  42.7  46.4  49.0  50.0  49.0  46.3  42.6  38.6  35.1  32.3  30.4
winter MW  25.2  25.4  25.8  26.4  27.3  28.4  29.9  31.4  32.7  33.7  34.0  33.7  33.0  32.0  31.2  30.8  30.8  31.0  30.8  30.0  28.8  27.4  26.4  25.6


### The three numbers that describe any facility's load

Peak, energy, and the ratio between them. Module 1 called the last one the **load factor**, and it is the single most useful number about a site: it tells you how much of your bill is energy and how much is capacity.


In [5]:
peak = max(park_summer.max(), park_winter.max())
energy_yr = park_summer.sum() * W_SUMMER + park_winter.sum() * W_WINTER
load_factor = energy_yr / (peak * 8760)

print(f'peak demand        {peak:9.2f} MW   (summer hour {int(park_summer.argmax())})')
print(f'annual energy      {energy_yr / 1000:9.1f} GWh')
print(f'load factor        {load_factor:9.3f}')


peak demand            50.00 MW   (summer hour 16)
annual energy          285.9 GWh
load factor            0.653


### The price, which is an input

Here is the whole of the boundary in one cell. **You are not going to compute this price.** It is a downloaded series - the day-ahead LMP at the site's node, as published.

It came out of the macro model in **M0B Part B**: someone ran a system model once, and the answer became a column in a file. That is called **soft-linking**, and it is the normal way a facility study gets its prices. From this cell until Part 7 the system model does not appear again.


In [6]:
# day-ahead LMP, $/MWh, as downloaded. M0B Part B produced these.
lmp_summer = np.array([
    52., 52., 44., 37., 31., 26., 26., 22., 22., 22.,  0.,  0.,
    22., 22., 26., 31., 37., 44., 52., 62., 90., 90., 90., 90.])

lmp_winter = np.array([
    37., 31., 26., 26., 22., 22., 22.,  0.,  0.,  0.,  0.,  0.,
     0.,  0.,  0., 22., 22., 26., 31., 37., 44., 44., 44., 52.])

print(f'summer  min ${lmp_summer.min():6.0f}   mean ${lmp_summer.mean():7.2f}   max ${lmp_summer.max():6.0f}')
print(f'winter  min ${lmp_winter.min():6.0f}   mean ${lmp_winter.mean():7.2f}   max ${lmp_winter.max():6.0f}')


summer  min $     0   mean $  41.25   max $    90
winter  min $     0   mean $  21.17   max $    52


### The tariff wrapped around it

A site does not pay the LMP. It pays the LMP plus everything the utility bundles on top. Module 2's facility coda called this out: **the number a behind-the-meter generator avoids is the retail rate, not the wholesale price.** This cell is where the two part company.


In [7]:
retail_summer = lmp_summer + ENERGY_ADDER + DELIVERY_VOL
retail_winter = lmp_winter + ENERGY_ADDER + DELIVERY_VOL

print(pd.DataFrame({'LMP summer': lmp_summer,
                    'retail summer': retail_summer,
                    'LMP winter': lmp_winter,
                    'retail winter': retail_winter}).T.to_string())


                 0     1     2     3     4     5     6     7     8     9     10    11    12    13    14    15    16    17    18    19     20     21     22     23
LMP summer     52.0  52.0  44.0  37.0  31.0  26.0  26.0  22.0  22.0  22.0   0.0   0.0  22.0  22.0  26.0  31.0  37.0  44.0  52.0  62.0   90.0   90.0   90.0   90.0
retail summer  82.0  82.0  74.0  67.0  61.0  56.0  56.0  52.0  52.0  52.0  30.0  30.0  52.0  52.0  56.0  61.0  67.0  74.0  82.0  92.0  120.0  120.0  120.0  120.0
LMP winter     37.0  31.0  26.0  26.0  22.0  22.0  22.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  22.0  22.0  26.0  31.0  37.0   44.0   44.0   44.0   52.0
retail winter  67.0  61.0  56.0  56.0  52.0  52.0  52.0  30.0  30.0  30.0  30.0  30.0  30.0  30.0  30.0  52.0  52.0  56.0  61.0  67.0   74.0   74.0   74.0   82.0


> **Predict before you run the next cell.** The site draws about 50 MW at peak and around 286 GWh a year. Write down two guesses: what the annual bill is, and what fraction of it you think comes from the demand charges rather than the energy.


In [8]:
bill_energy = ((park_summer * retail_summer).sum() * W_SUMMER
               + (park_winter * retail_winter).sum() * W_WINTER)
bill_demand = peak * DEMAND_PER_MW_YR
bill_total = bill_energy + bill_demand

print(f'volumetric charges   ${bill_energy / 1e6:9.2f} M/yr')
print(f'demand charges       ${bill_demand / 1e6:9.2f} M/yr   ({bill_demand / bill_total * 100:.0f}% of the bill)')
print(f'                     {"-" * 22}')
print(f'TOTAL                ${bill_total / 1e6:9.2f} M/yr')
print(f'blended rate         ${bill_total / energy_yr:9.2f} /MWh')


volumetric charges   $    17.28 M/yr
demand charges       $     6.20 M/yr   (26% of the bill)
                     ----------------------
TOTAL                $    23.48 M/yr
blended rate         $    82.15 /MWh


**That total is the number every later part of this notebook is measured against.** A facility study has one job: move it.

And look at the split. Roughly a quarter of the bill is bought with a single interval of demand, not with energy. A model that only counts $/MWh cannot see a quarter of the problem - which is exactly the mistake Part 2 is about to make on purpose.


---
# Part 2 - What would solar cost, and what is it worth?
### Module 2's tools, at the facility scale

Module 2 asked what capacity a *system* should build. The facility question is narrower: should **this site** put generation behind its own meter?

Start with what the site is physically allowed to build. Not a resource potential and not a land constraint - **roof area**, which is the constraint a system model has never once seen.


In [9]:
solar_max_mw = ROOF_MW
print(f'{ROOF_SQFT:,.0f} sq ft x {W_PER_SQFT:.0f} W/sq ft = {solar_max_mw:.1f} MW-dc')


1,200,000 sq ft x 10 W/sq ft = 12.0 MW-dc


### The output, and the derate that stops it being a brochure number

A clear-sky profile is an idealisation. Clouds, soiling, inverter losses and wiring take a bite out of every one of these hours, so the whole profile carries a derate. **That single factor moves the capacity factor from a number you could not defend to one you could.**


In [10]:
DERATE = 0.88     # clouds, soiling, inverter and wiring losses

solar_summer = (np.clip(np.sin(np.pi * (hours - 6.0) / 14.0), 0, 1) ** 1.15
                * 0.86 * DERATE)
solar_winter = (np.clip(np.sin(np.pi * (hours - 7.5) / 10.5), 0, 1) ** 1.15
                * 0.62 * DERATE)

solar_per_mw = (solar_summer.sum() * W_SUMMER
                + solar_winter.sum() * W_WINTER)
capacity_factor = solar_per_mw / 8760

print(pd.DataFrame({'summer p.u.': solar_summer.round(3),
                    'winter p.u.': solar_winter.round(3)}).T.to_string())
print()
print(f'output per MW-dc   {solar_per_mw:9,.0f} MWh/yr')
print(f'capacity factor    {capacity_factor:9.3f}')


              0    1    2    3    4    5    6      7      8      9     10     11     12     13     14     15     16     17    18     19   20   21   22   23
summer p.u.  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.134  0.290  0.440  0.57  0.671  0.735  0.757  0.735  0.671  0.570  0.440  0.29  0.134  0.0  0.0  0.0  0.0
winter p.u.  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.000  0.061  0.209  0.35  0.462  0.530  0.544  0.502  0.411  0.282  0.134  0.00  0.000  0.0  0.0  0.0  0.0

output per MW-dc       1,759 MWh/yr
capacity factor        0.201


### Annualise the capital, then divide - Module 2 slide 22 and 23

Identical arithmetic to the Houston CCGT, with two inputs changed: a 25-year life instead of 30, and no fuel term at all.


In [11]:
CRF_SOLAR = 0.07 * 1.07 ** 25 / (1.07 ** 25 - 1)
solar_annual = SOLAR_CAPEX * CRF_SOLAR + SOLAR_FOM
lcoe = solar_annual / solar_per_mw

print(f'CRF(7%, 25 yr)      {CRF_SOLAR:9.4f}')
print(f'CapEx x CRF        ${SOLAR_CAPEX * CRF_SOLAR:11,.0f} /MW-yr')
print(f'+ fixed O&M        ${solar_annual:11,.0f} /MW-yr')
print(f'/ {solar_per_mw:,.0f} MWh   ->  LCOE  ${lcoe:.2f} /MWh')


CRF(7%, 25 yr)         0.0858
CapEx x CRF        $     94,392 /MW-yr
+ fixed O&M        $    110,392 /MW-yr
/ 1,759 MWh   ->  LCOE  $62.75 /MWh


### Now the question this whole module exists for

You have a cost per MWh. **What do you compare it against?**

There are two candidates and they are not close:

- the **wholesale price** a merchant plant would *receive* for selling this energy into the market, and
- the **retail rate** this site *avoids paying* by not buying the energy at all.

Both have to be weighted by when the solar actually produces - a plant that generates at noon does not capture the 7 p.m. price. That weighting is why the next cell is longer than a simple average.

> **Predict:** how far apart are the two numbers? Within 20%? A factor of two? More?


In [12]:
wholesale_value = ((lmp_summer * solar_summer).sum() * W_SUMMER
                   + (lmp_winter * solar_winter).sum() * W_WINTER
                   ) / solar_per_mw
retail_value = ((retail_summer * solar_summer).sum() * W_SUMMER
                + (retail_winter * solar_winter).sum() * W_WINTER
                ) / solar_per_mw

print(f'LCOE, what it costs             ${lcoe:8.2f} /MWh')
print(f'wholesale, what a merchant gets ${wholesale_value:8.2f} /MWh')
print(f'retail, what this site avoids   ${retail_value:8.2f} /MWh')
print(f'ratio                            {retail_value / wholesale_value:8.2f} x')
print()
print(f'against wholesale   {lcoe - wholesale_value:+8.2f} /MWh   {"FAILS" if lcoe > wholesale_value else "CLEARS"}')
print(f'against retail      {lcoe - retail_value:+8.2f} /MWh   {"FAILS" if lcoe > retail_value else "CLEARS"}')


LCOE, what it costs             $   62.75 /MWh
wholesale, what a merchant gets $   16.72 /MWh
retail, what this site avoids   $   46.72 /MWh
ratio                                2.79 x

against wholesale     +46.03 /MWh   FAILS
against retail        +16.03 /MWh   FAILS


**Two things happened there, and only one of them is the one you were expecting.**

First, the retail rate is nearly three times the wholesale price. That is the distinction Module 2's facility coda said sinks more first-year solar business cases than any modelling error, and here it is worth $30/MWh. Using the wrong one would not have made the answer slightly wrong; it would have made it wrong by more than the whole margin.

Second - and this is the uncomfortable part - **the project still fails.** Even against the right price, the solar costs more per MWh than it saves.

A first-year analyst writes *"rooftop solar is uneconomic at this site"* and closes the file. That answer is defensible, well-reasoned, and wrong, and Part 4 is where you find out why. But you cannot skip to Part 4 - you have to know what the energy-only answer is before you can say what the demand charge is worth **on top of it**.


---
# Part 3 - Which limit actually binds?
### Module 3's tools, at the facility scale

Module 3 built DCOPF, susceptances, loop flow and congestion rent. A facility uses almost none of it. It models **one element** of a network: the connection between itself and the system.

There are three candidate limits on how much solar this site can have. Only one of them binds, and the useful skill is finding out which.


In [13]:
import_headroom = ICA_MW - peak

# the site may not export - a load interconnection agreement is not a
# generation one. So output can never exceed load in any lit hour.
lit = np.concatenate([solar_summer, solar_winter]) > 1e-6
load_all = np.concatenate([park_summer, park_winter])[lit]
pu_all = np.concatenate([solar_summer, solar_winter])[lit]
no_export_mw = (load_all / pu_all).min()

print(f'1. interconnection    {ICA_MW:8.0f} MW  -> {import_headroom:.0f} MW of import headroom')
print(f'2. roof area          {ROOF_MW:8.1f} MW-dc')
print(f'3. no export allowed  {no_export_mw:8.1f} MW-dc before the first spilled hour')
print()
print(f'binding limit: the ROOF, at {min(ROOF_MW, no_export_mw):.1f} MW-dc')


1. interconnection         600 MW  -> 550 MW of import headroom
2. roof area              12.0 MW-dc
3. no export allowed      52.9 MW-dc before the first spilled hour

binding limit: the ROOF, at 12.0 MW-dc


### What over-building would cost, if the roof were bigger

Worth computing even though it does not bind here, because it is the shape of the answer at every site that *does* have land.


In [14]:
rows = []
for mw in [12, 30, 45, 60, 75, 90]:
    gen_s, gen_w = solar_summer * mw, solar_winter * mw
    spilled = (np.maximum(gen_s - park_summer, 0).sum() * W_SUMMER
               + np.maximum(gen_w - park_winter, 0).sum() * W_WINTER)
    total = gen_s.sum() * W_SUMMER + gen_w.sum() * W_WINTER
    rows.append({'MW-dc': mw, 'spilled %': round(spilled / total * 100, 1),
                 'usable MWh/yr': round(total - spilled)})

print(pd.DataFrame(rows).set_index('MW-dc').to_string())


       spilled %  usable MWh/yr
MW-dc                          
12           0.0          21112
30           0.0          52780
45           0.0          79169
60           2.0         103410
75          11.4         116909
90          21.4         124528


**The interconnection does not bind, and finding that out is the result.**

That is worth saying plainly because it is the opposite of what students expect from a module about networks. The site has 550 MW of unused import headroom - the agreement was written for heavy industry that never arrived. For *this* question, at *this* size, the network is not the constraint and does not need to be modelled.

Two things follow, and they are both worth more than the calculation:

1. **Check which limit binds before you build the model, not after.** A week spent modelling the interconnection here would have produced a correct answer to a question nobody asked.
2. **Unused headroom is an asset.** Module 3's facility coda made this point and Part 7 collects on it: the reason investors buy this site is the 550 MW nobody is using.


---
# Part 4 - The charge the $/MWh comparison could not see
### Module 4's tools, at the facility scale

Part 2 valued a MWh of solar at the retail rate it avoids, and the project failed. But a solar array does not only avoid energy. If it happens to be producing during the interval that sets the demand charge, it also **reduces the peak** - and the peak is a quarter of this site's bill.

That value does not appear anywhere in a $/MWh comparison. It has to be computed separately and added.

> **Predict:** the site peaks at 4 p.m. in summer. Solar at 4 p.m. is past its best but nowhere near done. How many of the 50 MW do you think 12 MW of rooftop solar takes off the peak - and does the peak stay at 4 p.m.?


In [15]:
net_summer = park_summer - solar_summer * ROOF_MW
net_winter = park_winter - solar_winter * ROOF_MW
new_peak = max(net_summer.max(), net_winter.max())
shaved = peak - new_peak

print(pd.DataFrame({'load': park_summer.round(1),
                    'solar': (solar_summer * ROOF_MW).round(1),
                    'net': net_summer.round(1)}).T.to_string())
print()
print(f'peak before   {peak:8.2f} MW at hour {int(park_summer.argmax())}')
print(f'peak after    {new_peak:8.2f} MW at hour {int(net_summer.argmax())}')
print(f'shaved        {shaved:8.2f} MW')


         0     1     2     3     4     5     6     7     8     9     10    11    12    13    14    15    16    17    18    19    20    21    22    23
load   28.0  28.0  28.0  28.1  28.3  29.0  30.3  31.9  33.2  33.8  34.3  36.0  38.9  42.7  46.4  49.0  50.0  49.0  46.3  42.6  38.6  35.1  32.3  30.4
solar   0.0   0.0   0.0   0.0   0.0   0.0   0.0   1.6   3.5   5.3   6.8   8.1   8.8   9.1   8.8   8.1   6.8   5.3   3.5   1.6   0.0   0.0   0.0   0.0
net    28.0  28.0  28.0  28.1  28.3  29.0  30.3  30.3  29.7  28.5  27.5  27.9  30.1  33.6  37.5  41.0  43.2  43.7  42.9  41.0  38.6  35.1  32.3  30.4

peak before      50.00 MW at hour 16
peak after       43.75 MW at hour 17
shaved            6.25 MW


**The peak moved.** It was at hour 16 and it is now at hour 17, because solar falls off faster than the load does. That is not a curiosity - it is why you cannot estimate peak reduction as *"solar output at the old peak"*. The system re-peaks somewhere else, and the second peak is what you actually pay for.

Now put a price on it and stack the two streams. This is the same two-stream structure as Module 1 slide 24 and Module 4 slide 23 - the asset earns in more than one way and neither stream alone is the answer.


In [16]:
solar_energy = solar_per_mw * ROOF_MW
value_energy = solar_energy * retail_value
value_peak = shaved * DEMAND_PER_MW_YR
cost_solar = ROOF_MW * solar_annual

print(f'energy avoided       ${value_energy / 1e6:8.3f} M/yr   (${retail_value:.2f}/MWh x {solar_energy:,.0f} MWh)')
print(f'demand charge avoided${value_peak / 1e6:8.3f} M/yr   ({shaved:.2f} MW x ${DEMAND_PER_MW_YR:,.0f}/MW-yr)')
print(f'annual cost          ${cost_solar / 1e6:8.3f} M/yr')
print(f'{"-" * 46}')
print(f'NET on energy alone  ${(value_energy - cost_solar) / 1e6:8.3f} M/yr')
print(f'NET with both        ${(value_energy + value_peak - cost_solar) / 1e6:8.3f} M/yr')
print()
print(f'the demand charge is worth ${value_peak / solar_energy:.2f}/MWh of solar output,')
print(f'nearly as much again as the energy itself (${retail_value:.2f}) - and none of it')
print('appears anywhere in a levelised cost comparison.')


energy avoided       $   0.986 M/yr   ($46.72/MWh x 21,112 MWh)
demand charge avoided$   0.775 M/yr   (6.25 MW x $124,000/MW-yr)
annual cost          $   1.325 M/yr
----------------------------------------------
NET on energy alone  $  -0.338 M/yr
NET with both        $   0.437 M/yr

the demand charge is worth $36.72/MWh of solar output,
nearly as much again as the energy itself ($46.72) - and none of it
appears anywhere in a levelised cost comparison.


**The sign flipped.** Same array, same cost, same weather. The project went from losing money to making it, and nothing changed except that the model started counting a charge that was always on the bill.

This is the whole argument for the facility boundary. At system scale there is no demand charge - it is a cost-allocation artefact, not a physical quantity, and a macro model does not have one. At facility scale it is a quarter of the bill and the entire economics of the project.


### And now the problem the battery exists to solve

Everything above assumed the solar is producing during the interval that sets the peak. Look at what that assumption is actually worth.


In [17]:
print('solar output in the hours that could set the peak:')
for h in (15, 16, 17, 18):
    print(f'   hour {h}:  {solar_summer[h]:.3f} p.u.  ->  {solar_summer[h] * ROOF_MW:5.2f} MW on a clear day,  0.00 MW under cloud')
print()
print(f'one cloudy afternoon in the wrong hour and the whole '
      f'${value_peak / 1e6:.3f} M/yr is gone,')
print('because the demand charge is set by a single interval and billed '
      'for twelve months.')


solar output in the hours that could set the peak:
   hour 15:  0.671 p.u.  ->   8.06 MW on a clear day,  0.00 MW under cloud
   hour 16:  0.570 p.u.  ->   6.84 MW on a clear day,  0.00 MW under cloud
   hour 17:  0.440 p.u.  ->   5.27 MW on a clear day,  0.00 MW under cloud
   hour 18:  0.290 p.u.  ->   3.48 MW on a clear day,  0.00 MW under cloud

one cloudy afternoon in the wrong hour and the whole $0.775 M/yr is gone,
because the demand charge is set by a single interval and billed for twelve months.


Module 1 slide 23 put it this way: *you cannot shave a peak you did not see coming.* A solar array cannot promise to be there in a particular fifteen-minute interval four months from now. A battery can.

Which is why the battery in this notebook is **not** an arbitrage battery. Module 4 slide 23's 100 MW unit lives on the price spread. This one is bought for **duration and availability in one interval** - a completely different machine sized by a completely different number, which is exactly what Module 4's facility coda said.


---
# Part 5 - What it takes to actually get the equipment
### Module 4's second half, at the facility scale

Module 4 allocated material flows across a whole industry. This site sits at one node of that network and buys from it. The question is not where capacity should be built - it is what a delivered module costs at this dock, when it arrives, and what happens if the supplier fails.

Two quotes are on the desk. One is cheaper and comes from a single country; the other costs more and is diversified. The site's procurement policy caps any one supplier at 60% of a critical input - Chapter 22's single-source risk constraint, imposed by someone who has to keep a project on schedule rather than by a planner who likes diversity.


In [18]:
MODULE_W = 550.0           # watts per module
MODULE_SHARE = 0.30        # modules as a share of installed cost
A_PRICE, A_LEAD = 0.26, 20    # $/W ex-works, weeks, one country
B_PRICE, B_LEAD = 0.34, 8     # $/W ex-works, weeks, diversified
SINGLE_SOURCE_CAP = 0.60

n_modules = ROOF_MW * 1e6 / MODULE_W
blended = SINGLE_SOURCE_CAP * A_PRICE + (1 - SINGLE_SOURCE_CAP) * B_PRICE
capex_uplift = (blended - A_PRICE) * 1e6      # $/MW

print(f'modules needed        {n_modules:10,.0f} at {MODULE_W:.0f} W')
print(f'cheapest single source ${A_PRICE:9.3f} /W, {A_LEAD} weeks')
print(f'under the 60% cap      ${blended:9.3f} /W   (+${blended - A_PRICE:.3f}/W)')


modules needed            21,818 at 550 W
cheapest single source $    0.260 /W, 20 weeks
under the 60% cap      $    0.292 /W   (+$0.032/W)


### Feed it back into the LCOE

This is the step that gets skipped. A procurement constraint is not a footnote - it changes the capital cost, which changes the annualised cost, which changes the number the whole decision turned on in Part 2.


In [19]:
capex_with_cap = SOLAR_CAPEX + capex_uplift
annual_with_cap = capex_with_cap * CRF_SOLAR + SOLAR_FOM
lcoe_with_cap = annual_with_cap / solar_per_mw

print(f'CapEx    ${SOLAR_CAPEX / 1e6:6.3f} -> ${capex_with_cap / 1e6:.3f} /W   ({capex_with_cap / SOLAR_CAPEX - 1:+.1%})')
print(f'LCOE     ${lcoe:6.2f} -> ${lcoe_with_cap:.2f} /MWh')
print(f'still below the retail value it avoids? {lcoe_with_cap < retail_value + value_peak / solar_energy}')
print()
print(f'and the schedule: {A_LEAD} weeks gates {SINGLE_SOURCE_CAP:.0%} of the modules,')
print(f'so the array energises in phases, not on one day.')


CapEx    $ 1.100 -> $1.132 /W   (+2.9%)
LCOE     $ 62.75 -> $64.31 /MWh
still below the retail value it avoids? True

and the schedule: 20 weeks gates 60% of the modules,
so the array energises in phases, not on one day.


A 2.9% capital uplift does not change this answer - the margin from Part 4 is far bigger than that. **Say so explicitly in a report.** "We tested it and it does not bind" is a finding; silence is an omission a reviewer will find.

The lead time is the part that actually costs something, and it does not appear in any of the arithmetic above. It decides whether the array is earning during next summer's 4CP intervals or the ones after. **A year of the benefit you just computed is riding on a procurement decision, not on a modelling one.** Most graduates of this course will make that decision long before they make a modelling one.


---
# Part 6 - Now the streamlined version

Everything up to here was built one step at a time so you could see each decision. From this point the notebook wraps that construction in a function, and the rule about when you are allowed to do that applies: **you have already built every one of these components by hand, and we are about to run the same model four times** at different configurations. That is the reason, and it is the only acceptable one.

The model is Module 0's anatomy with nothing added:

| component | what it is here |
|---|---|
| `Bus` | the site, behind the meter |
| `Load` | the metered profile from Part 1 |
| `Generator` "grid" | the utility supply, priced at the retail tariff |
| `Generator` "solar" | the rooftop array, capped at the roof |
| `StorageUnit` | the battery, `max_hours=2` |

**Two arguments carry the whole model, and both are from Module 0 slides 40 to 43:**

- `p_nom_extendable=True` on solar and the battery turns this from a dispatch model into an investment model. That is Module 2's capacity expansion, applied to a roof.
- `capital_cost` on the **grid** generator is the trick worth remembering. The demand charge is a cost on your *highest* import, and an extendable generator's `p_nom` is exactly that: the largest value it ever has to supply. Put the demand charge in `capital_cost` and the solver prices your peak for you. No new machinery - a component you have used a dozen times, pointed at a different quantity.

`snapshot_weightings.objective` is Part 1's representative days, written in PyPSA: it tells the solver each summer hour stands for 165 days and each winter hour for 200.


In [20]:
index = pd.Index([f'S{h:02d}' for h in hours] + [f'W{h:02d}' for h in hours])
load_series = np.concatenate([park_summer, park_winter])
solar_series = np.concatenate([solar_summer, solar_winter])
retail_series = np.concatenate([retail_summer, retail_winter])
weights = np.array([W_SUMMER] * 24 + [W_WINTER] * 24, dtype=float)


def site_model(solar_max, batt_max, demand_charge=None, retail=None,
               capex_solar=None, load=None, ica=None):
    """The facility model. Every line appeared in Parts 0-5."""
    demand_charge = DEMAND_PER_MW_YR if demand_charge is None else demand_charge
    retail = retail_series if retail is None else retail
    capex_solar = solar_annual if capex_solar is None else capex_solar
    load = load_series if load is None else load
    ica = ICA_MW if ica is None else ica

    n = pypsa.Network()
    n.set_snapshots(index)
    n.snapshot_weightings.objective = weights   # representative days

    n.add('Bus', 'Site')
    n.add('Load', 'park', bus='Site', p_set=pd.Series(load, index=index))

    # the utility supply. capital_cost here IS the demand charge.
    n.add('Generator', 'grid', bus='Site', p_nom_extendable=True,
          p_nom_max=ica, capital_cost=demand_charge,
          marginal_cost=pd.Series(retail, index=index))

    if solar_max > 0:
        n.add('Generator', 'solar', bus='Site', p_nom_extendable=True,
              p_nom_max=solar_max, capital_cost=capex_solar,
              marginal_cost=0.0,
              p_max_pu=pd.Series(solar_series, index=index))

    if batt_max > 0:
        n.add('StorageUnit', 'battery', bus='Site', p_nom_extendable=True,
              p_nom_max=batt_max, capital_cost=BATT_ANNUAL,
              max_hours=BATT_HOURS,
              efficiency_store=RTE ** 0.5,
              efficiency_dispatch=RTE ** 0.5,
              cyclic_state_of_charge=True)

    n.optimize(solver_name=SOLVER, log_to_console=False)
    return n


CRF_BATT = 0.07 * 1.07 ** 15 / (1.07 ** 15 - 1)
BATT_ANNUAL = BATT_CAPEX_KWH * 1000.0 * BATT_HOURS * CRF_BATT
print(f'CRF(7%, 15 yr) {CRF_BATT:.4f}   battery ${BATT_ANNUAL:,.0f} /MW-yr')
print('Module 1 slide 24 quotes $65,877/yr for 1 MW / 2 MWh. Same number.')


CRF(7%, 15 yr) 0.1098   battery $65,877 /MW-yr
Module 1 slide 24 quotes $65,877/yr for 1 MW / 2 MWh. Same number.


### The check that earns the wrapper

Before trusting the function with anything, make it reproduce a number you already computed by hand. Run it with no solar and no battery: it should return exactly the bill from Part 1.

This is not ceremony. It is how you find out that the convenient version and the version you understand have quietly diverged.


In [21]:
do_nothing = site_model(0, 0)

print(f'solver objective   ${do_nothing.objective / 1e6:10.4f} M/yr')
print(f'hand-built bill    ${bill_total / 1e6:10.4f} M/yr')
print(f'peak it chose      {do_nothing.generators.p_nom_opt["grid"]:10.2f} MW  (hand: {peak:.2f})')

rel = abs(do_nothing.objective - bill_total) / bill_total
assert rel < 1e-6, f'the wrapper does not reproduce the hand-built bill ({rel:.2e})'
print()
print('the wrapper reproduces the hand-built bill exactly.')


Index(['Site'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 49 primals, 146 duals
Objective: 2.35e+07
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper were not assigned to the network.


solver objective   $   23.4847 M/yr
hand-built bill    $   23.4847 M/yr
peak it chose           50.00 MW  (hand: 50.00)

the wrapper reproduces the hand-built bill exactly.


> **Predict before the next cell.** You know from Part 4 that solar alone clears by about $0.44 M/yr once the demand charge is counted. How much battery do you think the solver buys, and does the total saving roughly double, or something less?


In [22]:
solar_only = site_model(ROOF_MW, 0)
both = site_model(ROOF_MW, 20.0)

out = []
for name, n in [('do nothing', do_nothing), ('solar only', solar_only),
                ('solar + battery', both)]:
    out.append({
        'case': name,
        'bill $M/yr': round(n.objective / 1e6, 3),
        'solar MW': round(n.generators.p_nom_opt.get('solar', 0.0), 2),
        'battery MW': round(n.storage_units.p_nom_opt.get('battery', 0.0)
                            if len(n.storage_units) else 0.0, 2),
        'peak MW': round(n.generators.p_nom_opt['grid'], 2),
        'saved $M/yr': round((do_nothing.objective - n.objective) / 1e6, 3)})

print(pd.DataFrame(out).set_index('case').to_string())


Index(['Site'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 98 primals, 244 duals
Objective: 2.30e+07
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



...

INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.07s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 243 primals, 582 duals
Objective: 2.28e+07
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


                 bill $M/yr  solar MW  battery MW  peak MW  saved $M/yr
case                                                                   
do nothing           23.485       0.0        0.00    50.00        0.000
solar only           23.048      12.0        0.00    43.75        0.437
solar + battery      22.794      12.0       10.56    38.46        0.691


Compare the `solar only` saving against the $0.437 M/yr you stacked by hand in Part 4. They agree, which means the hand calculation and the LP are the same model - one of them is just easier to explain to a client.

The battery adds about $0.25 M/yr on top - less than the solar did, and for a different reason. The array is buying energy it no longer has to purchase. The battery is buying **certainty about one interval**, which is worth something precisely because the array cannot promise it. Neither asset would clear on the other's argument.


### The finding

One more run, and it is the one to put on the last slide of a presentation. Set the demand charge to zero - keep everything else identical, including the energy prices - and ask the solver what it wants to build.

> **Predict.** Less solar, or none?


In [23]:
no_demand_charge = site_model(ROOF_MW, 20.0, demand_charge=0.0)

print(f'with the demand charge:  solar {both.generators.p_nom_opt["solar"]:5.2f} MW,  battery {both.storage_units.p_nom_opt["battery"]:5.2f} MW')
print(f'with NO demand charge:   solar {no_demand_charge.generators.p_nom_opt["solar"]:5.2f} MW,  battery {no_demand_charge.storage_units.p_nom_opt["battery"]:5.2f} MW')


Index(['Site'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.07s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 243 primals, 582 duals
Objective: 1.73e+07
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


with the demand charge:  solar 12.00 MW,  battery 10.56 MW
with NO demand charge:   solar  0.00 MW,  battery  0.00 MW


**Nothing. It builds nothing.**

Every megawatt of this project exists because of a line on a tariff sheet - a cost-allocation mechanism that has no counterpart anywhere in a macro model. The energy price never justified it and still does not.

Sit with that for a moment, because it is the whole course arriving from the other direction. In Module 0 you learned that lowering the boundary turns price from an output into an input. What Part 6 shows is the stronger version: **lowering the boundary also changes which quantities exist at all.** A demand charge is not a small term a macro model leaves out for tractability. It is not in there, because at system scale there is nothing for it to be.


---
# Part 7 - The boundary check
### Every number above rests on one assumption. Test it.

This whole notebook treated the price as exogenous. That was a **modelling assumption with an error bar**, not a fact, and M0B told you how to test it: put your load into a system model once and see whether the price moves.

At 50 MW it barely does - M0B measured a 3.6% own-bill error, which is smaller than the argument you would have about the weather year. But the investors are about to turn this site into a 500 MW data centre on the same interconnection, so re-run the test at the size that matters.

This is the one place the system model comes back. It is M0B Part B's network, rebuilt here rather than imported, so you can see that nothing was hidden.


In [24]:
# ---- the macro model from M0B Part B. Dallas node, wind, a merit order.
metro = (4000
         + 1000 * np.exp(-((hours - 19) ** 2) / 12.0)
         + 400 * np.exp(-((hours - 8) ** 2) / 6.0))
wind_pu = np.clip(0.30 + 0.55 * np.sin(np.pi * (hours - 2) / 16), 0, 1)

STACK = [(f'gas{i + 1:02d}', 600, c) for i, c in
         enumerate([22., 26., 31., 37., 44., 52., 62., 90.])]
STACK += [('peaker1', 300, 175.), ('peaker2', 300, 600.),
          ('scarcity', 300, 2000.)]

grid = pypsa.Network()
grid.set_snapshots(hours)
grid.add('Bus', 'Dallas')
grid.add('Bus', 'Site')
grid.add('Generator', 'wind', bus='Dallas', p_nom=5000, marginal_cost=0,
         p_max_pu=pd.Series(wind_pu, index=hours))
for nm, mw, c in STACK:
    grid.add('Generator', nm, bus='Dallas', p_nom=mw, marginal_cost=c)
grid.add('Load', 'metro', bus='Dallas', p_set=pd.Series(metro, index=hours))
grid.add('Link', 'interconnection', bus0='Dallas', bus1='Site',
         p_nom=ICA_MW, efficiency=1.0)
grid.add('Load', 'site', bus='Site', p_set=0.0)

grid.optimize(solver_name=SOLVER, log_to_console=False)
# '+ 0.0' turns the solver's -0.0 into 0.0. You will meet negative zero
# in solver output for the rest of your career; it means zero.
lmp_base = grid.buses_t.marginal_price['Dallas'].values + 0.0

# the series you 'downloaded' in Part 1 is this model's output. Prove it.
assert np.allclose(lmp_base, lmp_summer), 'the downloaded series has drifted'
print(f'price with no site   min ${lmp_base.min():6.0f}   max ${lmp_base.max():6.0f}')
print('and it matches the series Part 1 treated as downloaded data.')


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 312 primals, 672 duals
Objective: 1.67e+06
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


price with no site   min $     0   max $    90
and it matches the series Part 1 treated as downloaded data.


Now change **one number** - the site load - and re-solve. Nothing else about the network moves.


In [25]:
for site_mw in (50.0, 500.0):
    grid.loads.loc['site', 'p_set'] = site_mw
    grid.optimize(solver_name=SOLVER, log_to_console=False)
    lmp_now = grid.buses_t.marginal_price['Dallas'].values
    predicted = (lmp_base * site_mw).sum()
    actual = (lmp_now * site_mw).sum()
    moved = int((~np.isclose(lmp_now, lmp_base)).sum())
    print(f'{site_mw:6.0f} MW   price moved in {moved:2d} of 24 h   own-bill error {actual / predicted - 1:+7.1%}')


Index(['Dallas', 'Site'], dtype='object', name='name')


Index(['interconnection'], dtype='object', name='name')


Index(['0', '1'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - log_to_console: False


...

INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 312 primals, 672 duals
Objective: 2.32e+06
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


   500 MW   price moved in 19 of 24 h   own-bill error  +84.2%


**3.6% and 84%.** The same two numbers M0B reported, because it is the same test on the same model.

So: everything in Parts 1 to 6 is sound for the 50 MW park. The price-taker assumption holds, the retail tariff built on that price is real, and the recommendation stands.

**And none of it transfers to the data centre.** Not because the physics changed - the roof is the same roof - but because at 500 MW the price on which the entire tariff was built is a price this site would move. Three things break at once:

| | at 50 MW | at 500 MW |
|---|---|---|
| the price series | downloaded, and correct within 3.6% | wrong by 84%, because the load changes it |
| the roof | 12 MW against a 50 MW peak - material | 12 MW against a 500 MW peak - a rounding error |
| the interconnection | 550 MW of headroom, never binds | 100 MW of headroom, and it is the whole project |

The honest answer for the data centre is not a different number. It is **"this model does not apply; go back to M0B and hard-link."** Knowing when to say that is the difference between a defensible answer and a confident wrong one, and it is worth more in an interview than any result in this notebook.


---
# Part 8 - One sensitivity, because one assumption deserves it

This site is on an **indexed** tariff: its energy charge passes the wholesale price straight through, which is why the battery had a $600/MWh evening to arbitrage against. Plenty of sites are on a fixed rate instead.

That matters here because the summer series has real shape to work against: it bottoms out at $0/MWh in the middle of the day, when West Texas wind is covering the metro load, and reaches $90 in the evening. A flat rate deletes that spread entirely.

It is a contract term, not a physical fact, and it is exactly the kind of assumption a reviewer will ask about. So test it: hold everything else constant and flatten the energy charge.


In [26]:
FIXED_RATE = 45.0     # $/MWh flat energy charge, in place of the index
retail_fixed = np.full(48, FIXED_RATE + DELIVERY_VOL)

flat = site_model(ROOF_MW, 20.0, retail=retail_fixed)

print(f'indexed tariff   solar {both.generators.p_nom_opt["solar"]:5.2f} MW   battery {both.storage_units.p_nom_opt["battery"]:5.2f} MW')
print(f'fixed ${FIXED_RATE:.0f}/MWh    solar {flat.generators.p_nom_opt["solar"]:5.2f} MW   battery {flat.storage_units.p_nom_opt["battery"]:5.2f} MW')


Index(['Site'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.07s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 243 primals, 582 duals
Objective: 2.33e+07
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


indexed tariff   solar 12.00 MW   battery 10.56 MW
fixed $45/MWh    solar 12.00 MW   battery  3.65 MW


**The solar recommendation survives; the battery recommendation does not.** The array is bought with the demand charge, which the contract term did not touch. Most of the battery was bought with the evening price spread, which the contract term deleted.

Report it that way. *"Build the array regardless; revisit the battery when the supply contract is signed"* is a recommendation someone can act on. *"Build 12 MW of solar and 10.6 MW of storage"* is a number that will be wrong the moment procurement renegotiates.


---
# Part 9 - The answer, written down

You now have everything. Assemble it.

### The recommendation

**Yes - build the rooftop array. Treat the battery as a separate decision gated on the supply contract.** The array clears not on the energy it generates but on the demand charge it avoids, and it clears with enough margin to absorb a 60% single-source procurement cap.

### The five assumptions a reviewer will go after, and what you say

1. **Two representative days, not 8,760 hours.** A screening result. The direction is robust; the magnitude is not. Say so before they ask.
2. **The demand charge is modelled on the annual peak,** not month by month. This overstates the charge in both the baseline and the solar case, so the *saving* is close to right even though neither *bill* is.
3. **The model has perfect foresight of the peak interval. You do not.** This is the single largest optimism in the whole notebook. It is also the argument for the battery, so it cuts both ways - and Module 1 slide 23 already told you the mitigation is to curtail on every day that might contain a 4CP interval.
4. **Price is exogenous.** Tested in Part 7 and sound at 50 MW.
5. **The tariff is indexed.** Tested in Part 8; the solar answer survives and the battery answer does not.

### The interview version

This is the point of Module 5. Somebody asks what you can do. You have ninety seconds. Here is the shape:

> *"I sized on-site generation for a 50 MW industrial site. The thing that made it interesting is that the array failed on energy value - it cost about $63 a megawatt-hour and only avoided about $47 - so on a straight LCOE comparison you'd walk away. But a quarter of that site's bill was demand charges, and the array happened to be producing through the interval that set them. Counting that, it cleared by about $440,000 a year. I checked the procurement constraint didn't eat the margin, and I checked that the site was small enough for the published price to still be valid - at ten times the load it wouldn't have been, and I'd have had to model the market instead of downloading it."*

Every clause in that paragraph is a module. Nobody has to know that.


### Your turn

Write your own version and put it in `RECOMMENDATION` below. Not a summary of this notebook - **your** site, or this one with a number you disagree with and can defend.

Three sentences. What you would do, what it is worth, and the one assumption that would change your mind.


In [ ]:
# RECOMMENDATION = """
# ...three sentences...
# """

if 'RECOMMENDATION' not in globals():
    raise NameError(
        'Write your recommendation in the cell above and uncomment it.\n'
        'Three sentences: what you would do, what it is worth, and the one\n'
        'assumption that would change your mind. This notebook will not\n'
        'write it for you - that sentence is the deliverable, and it is\n'
        'the part a hiring manager will actually read.')

words = len(RECOMMENDATION.split())
print(f'{words} words.')
print('Long enough.' if 25 <= words <= 140 else
      'Too short to be a recommendation.' if words < 25 else
      'Too long. A recommendation that needs 140 words is not one yet.')


---
## What Module 5 was for

1. **One question, answered end to end.** Not five exercises - one decision, with every module contributing a piece and none of them sufficient alone.
2. **The answer came from the tariff, not the technology.** Set the demand charge to zero and the model builds nothing. The most important input in the whole study was a line on a bill.
3. **A model that says no is doing its job.** Part 2's failure was not a wrong answer to be fixed. It was the right answer to a narrower question, and knowing it was narrower is the skill.
4. **The boundary was checked, not assumed.** Part 7 is the difference between a defensible number and a confident wrong one, and it takes one extra solve.
5. **Nothing new was needed.** Every component came from Modules 0 to 4. If that feels anticlimactic, notice what it means: you already have enough to do professional work, and have had for some weeks.

### Where this goes

This notebook is the scaffold for the capstone. The capstone changes the site and the question; the structure - state the boundary, get the baseline, cost the option, find the binding limit, count every revenue stream, price the supply chain, hand it to the solver, test the assumption, write the paragraph - does not.

### Sources and notes
- The Metroplex Industrial Park, its investors and its 600 MW agreement are invented. The pattern of buying industrial sites for their existing interconnection is not.
- The price series is M0B Part B's output, which is a teaching stand-in for ERCOT. Its *shape* - a long cheap base and a thin, very expensive tail - is the realistic feature that matters.
- The 4CP transmission charge and the $300/kWh battery are the same figures used on Module 1 slide 24 and Module 4 slide 23; the tariff's other components are representative of a large commercial ERCOT schedule and should be replaced with your own site's rate sheet.
- Solar capacity factor here comes from a clear-sky profile with a single derate. A real study uses NSRDB or PVWatts for the site's own coordinates.
